In [ ]:
import pandas as pd
import numpy as np
import re

In [50]:
def check_integrity(dataframe, filter_regex):
    """
    Funcion que checa la integridad de los datos de actividad para cada medio.
    Si hay multiples métricas de actividad itera por cada una y lo contrasta contra el spend. 
    Regresa un diccionario con cada columna de actividad y su % de integridad. 
    """
    # Frame con solo las columnas de cada medio declarado
    working_frame = dataframe.filter(regex=filter_regex)

    # Array para la variable spend
    spend_array = working_frame.filter(like='Spend').notna().to_numpy()
    array_lenght = spend_array.shape[0]

    # Frame para las variables de actividad (Effort)
    working_frame_effort = working_frame.filter(like='Effort')

    # Listado de columnas de actividad (Effort)
    working_frame_effort_columns = working_frame_effort.columns

    # Diccionario para almacenar resultados
    results = {}

    # Iterar por cada columna 
    for effort_column in working_frame_effort_columns:

        # Inicializar la suma 
        sum = 0

        # Obtener el array de la columna de actividad a comparar
        effort_array = working_frame_effort.filter(like=effort_column).notna().to_numpy()
        
        # Hacer la comparacion elemento a elemento 
        for i in range(array_lenght):
            if spend_array[i] == effort_array[i]:
                sum += 1

        # Añadir al diccionario de resultados el ratio de coincidencias / total elementos, es decir % de integridad
        results[effort_column] = sum/array_lenght

    return results

In [51]:
def media_integrity_checker(dataframe, hierarchy_df):
    """
    Funcion que itera por cada medio para identificar el grado de integridad de cada columna de actividad vs su respectivo spend.
    Esta funcion identifica los medios a evaluar y usa la funcion check_integrity para obtener los resultados de integridad de cada columna de actividad. 
    Regresa un dataframe con la integridad por medio. 
    """

    # Crear el listado de medios a evaluar 
    media_list = hierarchy_df[hierarchy_df['L0'] == 'Media'][['L2','L3', 'L4']]
    media_list.drop_duplicates(inplace=True)
    media_list = media_list.agg('::'.join, axis=1).to_list()

    # Diccionario que guarda los resultados
    media_results = {}

    # Itera por cada medio para obtener los resultados de todas las columnas de actividad 
    for media in media_list:
        media_results[media] = check_integrity(dataframe, media)

    return pd.DataFrame(media_results)


In [52]:
def separator_replacer(input_string):
    space_pattern = r'::'
    space_remover = r'\s'

    str_noSpaces = re.sub(pattern=space_remover, repl='', string=input_string)

    return re.sub(pattern=space_pattern, repl='_', string=str_noSpaces)


In [53]:
def output_media_column_namer(columns_list):
    """
    Funcion que renombra columnas las variables por cada medio
    """

    trunc_pattern = r'(^[\w\s]*::[\w\s]*::|::[\w\s]*$)'
    space_pattern = r'::'

    output_list = []

    for column_name in columns_list:
        truncated_column_name = re.sub(pattern=trunc_pattern, repl='', string=column_name)
        final_column_name = re.sub(pattern=space_pattern, repl=' ', string=truncated_column_name)

        output_list.append(final_column_name)
    
    return output_list

In [54]:
def per_media_df_generator(dataframe, hierarchy_df):
    """
    Genera un dataframe con la data de cada medio. 
    Regresa tuplas del nombre de la hoja y el dataframe correspondiente 
    """

    # Crear el listado de medios a evaluar 
    media_list = hierarchy_df[hierarchy_df['L0'].apply(lambda x: 'media' in x.lower())][['L2','L3', 'L4']]
    media_list.drop_duplicates(inplace=True)
    media_list = media_list.agg('::'.join, axis=1).to_list()

    # Lista de dataframes
    media_dataframes = []

    for media in media_list:
        temp_df = dataframe.filter(regex=media)
        temp_df.columns = output_media_column_namer(temp_df.columns)
        temp_df.loc[:,'Period'] = dataframe.loc[:,'Period']
        temp_df = temp_df.set_index('Period')

        media_dataframes.append((separator_replacer(media), temp_df))

    return media_dataframes

In [55]:
def cost_per_activity_creator(list_of_per_media_frames):
    """
    Funcion que toma el output de la funcion per_media_df_generator un listado de tuplas, en cada tupla esta el nombre y el dataframe con las variables por medio.
    Regresa un DataFrame con el costo por actividad de cada medio. 
    """

    # Iteramos por cada indice del listado de tuplas 
    for i_media in range(len(list_of_per_media_frames)):

        # Identificamos el nombre de la columna donde esta el spend
        spend_col_name = [name for name in list_of_per_media_frames[i_media][1].columns if 'Spend' in name]

        if len(spend_col_name) == 0:
            continue

        # Si es el primero creamos el dataframe result con la división de todo el frame por la columna del spend 
        if i_media == 0:
            result_df = list_of_per_media_frames[i_media][1]
            result_df = result_df.apply(lambda col: col / result_df.loc[:,spend_col_name[0]], axis=0)
        
        # Si es una iteracion subsecuente hacemos lo mismo pero vamos haciendo append con el dataframe resultado 
        else:
            temp_df = list_of_per_media_frames[i_media][1]
            temp_df = temp_df.apply(lambda col: col / temp_df.loc[:,spend_col_name[0]], axis=0)
            result_df = pd.merge(result_df, temp_df, left_index=True, right_index=True)
    
    # Hacemos limpieza identificando las columnas de spend y eliminandolas del frame final 
    spend_cols = [name for name in result_df.columns if 'Spend' in name]
    result_df.drop(columns=spend_cols, inplace=True)

    return result_df

In [56]:
def topline_sales_revenue_generator(dataframe_stacked):

    # Filtramos por los datos de ventas en revenue
    filtered_dataframe_sales_rev = dataframe_stacked[(dataframe_stacked['L0'] == 'Sales') & (dataframe_stacked['Metric'] == '$')]

    # Creamos el crosstab de las ventas en revenue por periodo 
    sales_revenue_topline = pd.crosstab(
        index=filtered_dataframe_sales_rev['L1'], 
        columns=filtered_dataframe_sales_rev['Breaks'], 
        values=filtered_dataframe_sales_rev['value'], 
        aggfunc='sum').round(0)

    # Creamos el crosstab del cambio porcentual periodo a periodo 
    sales_revenue_topline_changeYoY = (sales_revenue_topline.T.diff(1) / sales_revenue_topline.T.shift(1)).T.round(2)

    # Lo unimos en una sola tabla
    sales_revenue_dataframe = pd.merge(
        left=sales_revenue_topline, 
        right=sales_revenue_topline_changeYoY,
        left_on='L1',
        right_on='L1',
        suffixes=('',' change vs LY'))
    
    return sales_revenue_dataframe


In [57]:
def topline_sales_volume_generator(dataframe_stacked):
    """
    Funcion que genera tablas de volumen de ventas por cada metrica de volumen que tenemos. 
    Genera una lista con tuplas del nombre de pestaña y el dataframe correspondiente. 
    """

    # Filtramos el dataframe principal para tener solo datos de ventas y las metricas que no es revenue. 
    filtered_dataframe_sales_vol = dataframe_stacked[(dataframe_stacked['L0'] == 'Sales') & (dataframe_stacked['Metric'] != '$')]

    # Creamos la lista para capturar la salida de la función 
    sales_vol_dataframe_list = []

    # Por cada metrica ... 
    for metric in filtered_dataframe_sales_vol['Metric'].unique():

        # Volvemos a filtrar el frame por esa submetrica 
        filtered_dataframe_vol_submetric = filtered_dataframe_sales_vol[filtered_dataframe_sales_vol['Metric']==metric]

        # Creamos el crosstab del volumen por periodo (break)
        sales_metric_topline = pd.crosstab(
            index=filtered_dataframe_vol_submetric['L1'], 
            columns=filtered_dataframe_vol_submetric['Breaks'], 
            values=filtered_dataframe_vol_submetric['value'], 
            aggfunc='sum').round(0)

        # Creamos el crosstab del cambio porcentual por periodo 
        sales_metric_topline_changeYoY = (sales_metric_topline.T.diff(1) / sales_metric_topline.T.shift(1)).T.round(2)

        # Los unimos en usa sola tabla
        topline_sales_vol = pd.merge(
            left=sales_metric_topline, 
            right=sales_metric_topline_changeYoY,
            left_on='L1',
            right_on='L1',
            suffixes=(f' ({metric})',f' change vs LY ({metric})'))
        
        # Lo añadimos a nuestra lista 
        sales_vol_dataframe_list.append(
            (f'TL Sales Vol {metric}', topline_sales_vol)
        )
    
    # Regresa la salida 
    return sales_vol_dataframe_list 

In [58]:
def topline_efforts_avg_generator(dataframe_stacked):    
    other_efforts_data = dataframe_stacked[(dataframe_stacked['L0'] != 'Media') & (dataframe_stacked['Aggregation'] == 'Average')]

    sheet_list = []

    for effort in other_efforts_data['L0'].unique():

        other_efforts_data_subeffort = other_efforts_data[other_efforts_data['L0'] == effort]

        topline_avg_efforts_cross = pd.crosstab(
            index=other_efforts_data_subeffort['L1'],
            columns=other_efforts_data_subeffort['Breaks'],
            values=other_efforts_data_subeffort['value'],
            aggfunc='mean'
        )

        topline_avg_efforts_cross_changeYoY = (topline_avg_efforts_cross.T.diff(1) / topline_avg_efforts_cross.T.shift(1)).T.round(2)

        topline_avg_efforts_cross = pd.merge(
            left=topline_avg_efforts_cross,
            right=topline_avg_efforts_cross_changeYoY,
            left_on='L1',
            right_on='L1',
            suffixes=('', ' change vs LY')
        )

        sheet_list.append((f'{effort}', topline_avg_efforts_cross))

    return sheet_list

In [59]:
def topline_spend_generator(dataframe_stacked):

    # Filtramos por los datos de ventas en revenue
    filtered_dataframe_media = dataframe_stacked[(dataframe_stacked['L0'] == 'Media') & (dataframe_stacked['Metric Type'] == 'Spend')]

    # Creamos una columna con cada medio
    filtered_dataframe_media['L2:L4'] = filtered_dataframe_media.loc[:,('L2','L3','L4')].agg('-'.join, axis=1)

    # Creamos el crosstab de las ventas en revenue por periodo 
    media_spend_dataframe = pd.crosstab(
        index=filtered_dataframe_media['L2:L4'],
        columns=filtered_dataframe_media['Breaks'],
        values=filtered_dataframe_media['value'],
        aggfunc='sum').round(0)

    # Creamos el crosstab del cambio porcentual periodo a periodo 
    media_spend_dataframe_changeYoY = (media_spend_dataframe.T.diff(1) / media_spend_dataframe.T.shift(1)).T.round(2)

    # Creamos el crosstab del market share
    media_spend_dataframe_share = (media_spend_dataframe / media_spend_dataframe.sum()).round(2)

    # Lo unimos en una sola tabla
    media_spend_dataframe = pd.merge(
        left=media_spend_dataframe, 
        right=media_spend_dataframe_changeYoY,
        left_on='L2:L4',
        right_on='L2:L4',
        suffixes=('',' change vs LY'))
    
    media_spend_dataframe = pd.merge(
        left=media_spend_dataframe, 
        right=media_spend_dataframe_share,
        left_on='L2:L4',
        right_on='L2:L4',
        suffixes=('',' Spend Share'))
    
    return media_spend_dataframe

In [60]:
# Carga de datasets 
data = pd.read_excel("Input.xlsx", sheet_name="DATA") 
hierarchy = pd.read_excel("Input.xlsx", sheet_name='HIERARCHY')

# Anticheat: convertir zeros a NAs 
data = data.replace(0, np.nan)

# Apilamiento del set de datos 
stacked_data = data.melt(id_vars=('Period', 'Breaks')).copy()

# Union con jerarquia 
stacked_data = pd.merge(stacked_data, hierarchy, left_on='variable', right_on='Variable Name')

In [61]:
# Generación de descriptivos para identificar principales problemas de datos 

summary_describe = data.drop('Period', axis=1).describe().T

summary_describe['IQR'] = summary_describe['75%'] - summary_describe['25%']
summary_describe['Integrity'] = summary_describe['count'] / summary_describe['count'].max() * 100
summary_describe['Normalized Mean-Average'] = (summary_describe['mean'] - summary_describe['50%']) / summary_describe['mean']
summary_describe['Normalized STD-IQR'] = summary_describe['std'] - summary_describe['IQR'] / summary_describe['mean']

sheet_describe = summary_describe.round(2)

# Correlaciones 
sheet_correlations = data.fillna(0).corr(numeric_only=True)


In [ ]:
flag_media = input('Is media data available? (Y=Yes, N=No): ')

if flag_media == 'Y':

    # Diagnostico de integridad
    sheet_media_integrity = media_integrity_checker(data, hierarchy).round(3)

    # Frame de costo por actividad
    sheets_media_data = per_media_df_generator(data, hierarchy)
    sheet_cost_per_act = cost_per_activity_creator(sheets_media_data)
    sheet_describe_cpa = sheet_cost_per_act.describe().T.round(2)

    # Creamos la tabla resumen de spend por media 
    sheet_topline_media_spend = topline_spend_generator(stacked_data)

In [ ]:
flag_sales = input('Is sales data available? (Y=Yes, N=No): ')

if flag_sales == 'Y':

    # Creamos la tabla de resumen de ventas en revenue
    sheet_topline_sales_rev = topline_sales_revenue_generator(stacked_data)

    # Creamos las tablas de resumen de ventas por cada metrica de volumen
    sheets_topline_sales_vol = topline_sales_volume_generator(stacked_data)

flag_oth = input('Is price / distr / other average agg variables available? (Y=Yes, N=No): ')

if flag_oth == 'Y':
    
    # Creamos las tablas de resumen de los otros esfuerzos que se agregan por promedio (precio, distribucion, etc)
    sheets_topline_efforts_avg = topline_efforts_avg_generator(stacked_data)

In [64]:
# Exportar los analisis en un Excel 
with pd.ExcelWriter('Output.xlsx') as writer:
    
    sheet_describe.to_excel(writer, sheet_name='Describe')

    if flag_media == 'Y':
        sheet_media_integrity.to_excel(writer, sheet_name='Media integrity')

    sheet_correlations.to_excel(writer, sheet_name='Correlations')

    if flag_media == 'Y':
        sheet_cost_per_act.to_excel(writer, sheet_name='Cost per activity')
        sheet_describe_cpa.to_excel(writer, sheet_name='Descriptivos CPA')

    if flag_sales == 'Y':
        sheet_topline_sales_rev.to_excel(writer, sheet_name='TL Sales $')

        # Hojas de ventas en volumen 
        for sheet in range(len(sheets_topline_sales_vol)):
            sheets_topline_sales_vol[sheet][1].to_excel(writer, sheet_name=sheets_topline_sales_vol[sheet][0])

    if flag_media == 'Y':
        sheet_topline_media_spend.to_excel(writer, sheet_name='TL Media Spend')

    if flag_oth == 'Y':
        # Hojas de esfuerzos en promedio  
        for sheet in range(len(sheets_topline_efforts_avg)):
            sheets_topline_efforts_avg[sheet][1].to_excel(writer, sheet_name=sheets_topline_efforts_avg[sheet][0])
    
    if flag_media == 'Y':
        # Hojas de data por medio 
        for sheet in range(len(sheets_media_data)):
            sheets_media_data[sheet][1].to_excel(writer, sheet_name=sheets_media_data[sheet][0])

    data.to_excel(writer, sheet_name='DATA')
    stacked_data.to_excel(writer, sheet_name='STACKED DATA')